In [44]:
from coregiver import input_maker
import os # Needed for checking file existence and content
import numpy as np
import pandas as pd
from astropy import units as u

# Cham bimodal 

In [45]:
cham_bi_embs = np.loadtxt("/Users/honeyeah/Codes/data/cham_bi_tajer/cham_bi_tajer_3/cham_bimodal_emb_samples_sep3.txt")
cham_bi_pls = np.loadtxt("/Users/honeyeah/Codes/data/cham_bi_tajer/cham_bi_tajer_3/cham_bimodal_pl_samples_sep3.txt")
emb_masses = np.ones(14) * 2.8e-7
pl_masses = np.ones(140) * 2.8e-8 
cham_semis = np.concatenate([cham_bi_embs, cham_bi_pls])
cham_masses = np.concatenate([emb_masses, pl_masses])

cham_bi_df = pd.DataFrame(columns=['m', 'a'])
cham_bi_df['a'] = cham_semis
cham_bi_df['m'] = cham_masses

In [46]:
def get_mass_inner(masses, semis, bound):
    m_in = 0
    for i in range(len(semis)):
        if semis[i] <= bound:
            m_in += masses[i]

    return m_in

def get_tot_mass(masses):
    return sum(masses)

def get_cmf_inner(cmf_tot, x, m_in, m_tot):
    return cmf_tot*((1-x) + x*(m_tot/m_in))

def get_cmf_outer(x, cmftot):
    return (1-x)*cmftot

In [47]:
percentages = [0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.50, 0.55]
bounds = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]
base_cmf = 0.33

In [48]:
cmf_grid = pd.DataFrame(columns=['bounds', 'percentages', 'inner_cmf', 'outer_cmf'])
grid_index = 0

for bound in bounds:
    for perc in percentages:
        m_in = get_mass_inner(cham_bi_df['m'], cham_bi_df['a'], bound=bound)
        mtot = get_tot_mass(cham_bi_df['m'])
        cmf_inner = get_cmf_inner(base_cmf, perc, m_in, mtot)
        cmf_outer = get_cmf_outer(perc, base_cmf)
        # Skip if any CMF > 1
        if cmf_inner > 1 or cmf_outer > 1:
            print(f"⚠️ Skipping: Bound = {bound}, perc. = {perc} → CMF > 1 (CMF_in = {cmf_inner:.4}, CMF_out = {cmf_outer:.4})")
            continue

        input_maker.generate_input_step_function(len(cham_bi_df),
            cham_bi_df['m'], cham_bi_df['a'], cmf_inner, cmf_outer, bound,
            f"step_functions/cham_bi_tajer_step_b{bound}_p{perc}.txt"
        )

        cmf_grid.loc[grid_index] = {
            'bounds': bound,
            'percentages': perc,
            'inner_cmf': cmf_inner,
            'outer_cmf': cmf_outer
        }

        print(f"Bound = {bound}, perc. of Fe trans. = {perc}, CMF_in = {cmf_inner:.4}, CMF_out = {cmf_outer:.4}")
        grid_index += 1

Output file: step_functions/cham_bi_tajer_step_b0.5_p0.01.txt
Successfully generated 'step_functions/cham_bi_tajer_step_b0.5_p0.01.txt' with 154 entries.
Bound = 0.5, perc. of Fe trans. = 0.01, CMF_in = 0.5577, CMF_out = 0.3267
Output file: step_functions/cham_bi_tajer_step_b0.5_p0.02.txt
Successfully generated 'step_functions/cham_bi_tajer_step_b0.5_p0.02.txt' with 154 entries.
Bound = 0.5, perc. of Fe trans. = 0.02, CMF_in = 0.7854, CMF_out = 0.3234
⚠️ Skipping: Bound = 0.5, perc. = 0.03 → CMF > 1 (CMF_in = 1.013, CMF_out = 0.3201)
⚠️ Skipping: Bound = 0.5, perc. = 0.04 → CMF > 1 (CMF_in = 1.241, CMF_out = 0.3168)
⚠️ Skipping: Bound = 0.5, perc. = 0.05 → CMF > 1 (CMF_in = 1.469, CMF_out = 0.3135)
⚠️ Skipping: Bound = 0.5, perc. = 0.1 → CMF > 1 (CMF_in = 2.607, CMF_out = 0.297)
⚠️ Skipping: Bound = 0.5, perc. = 0.15 → CMF > 1 (CMF_in = 3.746, CMF_out = 0.2805)
⚠️ Skipping: Bound = 0.5, perc. = 0.2 → CMF > 1 (CMF_in = 4.884, CMF_out = 0.264)
⚠️ Skipping: Bound = 0.5, perc. = 0.25 → CMF

In [49]:
cmf_grid.to_csv("cham_bi_tajer_cmf_grid.csv", index=False)

In [61]:
n_embs = 14
n_pls= 140

cmf_linears_embs = np.linspace(0.9, 0.1, n_embs)
cmf_linears_pls = np.linspace(0.9, 0.1, n_pls)

cmf_linears = np.concatenate([cmf_linears_embs, cmf_linears_pls])

In [62]:
def generate_input_linear(num_particles, particle_mass, particle_cmfs, output_filename):
    particle_mass = np.array(particle_mass)
    with open(output_filename, 'w') as f:
        for i in range(num_particles):
            particle_hash = i + 1
            mass = particle_mass if particle_mass.size == 1 else particle_mass[i]
            particle_cmf = particle_cmfs[i]
            f.write(f"{particle_hash}\t{mass:.8e}\t{particle_cmf:.8e}\n")

        print(f"Successfully generated '{output_filename}' with {num_particles} entries.")

In [63]:
generate_input_linear(n_embs + n_pls, cham_bi_df['m'], cmf_linears, "linear_cham_bi.txt")

Successfully generated 'linear_cham_bi.txt' with 154 entries.


In [64]:
np.average(cmf_linears)

np.float64(0.5)

In [71]:

n_embs = 14
n_pls = 140
y_min, y_max = 0.1, 0.8
target_mean = 0.33

# total number of elements
n_total = n_embs + n_pls

# exponential x grid
x_embs = np.linspace(0, 1, n_embs)
x_pls = np.linspace(0, 1, n_pls)

# exponential decline (raw)
k = 3.0  # controls steepness; can adjust
y_embs = y_min + (y_max - y_min) * np.exp(-k * x_embs)
y_pls = y_min + (y_max - y_min) * np.exp(-k * x_pls)

# rescale so that mean is exactly target_mean
current_mean_embs = np.mean(y_embs)
current_mean_pls = np.mean(y_pls)
print(current_mean_embs)
print(current_mean_pls)
#y = (y - np.mean(y)) * (target_mean / current_mean) + target_mean

0.3330370015903883
0.32276567517967863


In [72]:
cmfs = np.concatenate([y_embs, y_pls])

In [73]:
generate_input_linear(154, cham_bi_df['m'], cmfs, "exp_cham_bi.txt")

Successfully generated 'exp_cham_bi.txt' with 154 entries.
